# Test `run_volumes_v3k` — per-class P3(k) mean widths

Validates the new `build_all_polytopes_per_class` and `run_volumes_v3k` pipeline.

Uses **3 random directions** on a single MLP sample to check:
- `width_correct`  = mean width of P2
- `widths_both[k]` = mean width of P3(k) for each class k
- Non-additivity: Σ_k V3(k) ≈ 2 × V2  (for q-incorrect sample)
- Old GACC = V3(c) / V2  vs.  New GACC = V3(c) / Σ_k V3(k)

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT)
print("Project root:", ROOT)

import torch
import numpy as np

from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.optim.build_polytopes import (
    build_all_polytopes,            # old API — must still work unchanged
    build_all_polytopes_per_class,  # new API
    check_polytope_membership,
)

Project root: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


## Load model and sample

In [12]:
torch.manual_seed(42)

model = FashionMLP_Large()
model.load_state_dict(
    torch.load(os.path.join(ROOT, "checkpoints/fashion_mlp_best.pth"),
               weights_only=True, map_location="cpu")
)
model.eval()

dataset = torch.load(
    # os.path.join(ROOT, "data/fashionMNIST_correct_mlp.pt"),
    os.path.join(ROOT, "data/qmodel_qincorrect_mlp_b4.pt"),
    weights_only=False
)

SAMPLE_IDX = 0        # change to test other samples
BITS_TEST  = [4, 8]   # small bits grid for speed
N_DIR      = 2        # very small — just for functional testing

x, c = dataset[SAMPLE_IDX]
c = int(c)
x_batch = x.flatten().unsqueeze(0)   # (1, 784)
x_vec   = x_batch.view(-1)           # (784,)

qmodels_dict = {b: quantize_model(model, bits=b).eval() for b in BITS_TEST}

# Model and qmodel predictions
with torch.no_grad():
    pred_model  = model(x_batch).argmax(dim=1).item()
    pred_qmodel = {b: qm(x_batch).argmax(dim=1).item() for b, qm in qmodels_dict.items()}

print(f"Sample idx      : {SAMPLE_IDX}")
print(f"True label c    : {c}")
print(f"Model pred      : {pred_model}")
print(f"qModel preds    : {pred_qmodel}")
q_correct = {b: (pred_qmodel[b] == c) for b in BITS_TEST}
print(f"q-correct?      : {q_correct}")

Sample idx      : 0
True label c    : 6
Model pred      : 6
qModel preds    : {4: 2, 8: 6}
q-correct?      : {4: False, 8: True}


## 1 — Backward-compatibility check

The old `build_all_polytopes` must still return `{bits: (A_correct, b_correct, A_both, b_both)}`.

In [13]:
A_base_old, b_base_old, pd_old = build_all_polytopes(model, qmodels_dict, x_batch, c)

print("Old API — build_all_polytopes:")
print(f"  A_base : {tuple(A_base_old.shape)}")
for bits, (A_c, b_c, A_b, b_b) in pd_old.items():
    in_c = check_polytope_membership(A_c, b_c, x_vec).item()
    in_b = check_polytope_membership(A_b, b_b, x_vec).item()
    print(f"  bits={bits:2d}  A_correct={tuple(A_c.shape)}  A_both={tuple(A_b.shape)}  "
          f"x∈P2={in_c}  x∈P3(c)={in_b}")
print("\n→ Old API intact.")

Old API — build_all_polytopes:
  A_base : (1929, 784)
  bits= 4  A_correct=(3849, 784)  A_both=(3858, 784)  x∈P2=True  x∈P3(c)=False
  bits= 8  A_correct=(3849, 784)  A_both=(3858, 784)  x∈P2=True  x∈P3(c)=True

→ Old API intact.


## 2 — New API: `build_all_polytopes_per_class`

In [14]:
A_base, b_base, pd_new = build_all_polytopes_per_class(model, qmodels_dict, x_batch, c)

print("New API — build_all_polytopes_per_class:")
print(f"  A_base : {tuple(A_base.shape)}")

for bits, (A_c, b_c, per_class) in pd_new.items():
    print(f"\n  bits={bits}")
    print(f"    A_correct : {tuple(A_c.shape)}")
    print(f"    n_classes : {len(per_class)}")
    for k, (A_k, b_k) in per_class.items():
        in_k = check_polytope_membership(A_k, b_k, x_vec).item()
        marker = " ← c (true class)" if k == c else (" ← k* (q-pred)" if pred_qmodel[bits] == k and not q_correct[bits] else "")
        print(f"    P3({k}): {tuple(A_k.shape)}  x∈P3({k})={in_k}{marker}")

# Sanity: A_correct same as old; P3(c) same as old A_both
for bits in BITS_TEST:
    A_c_old, b_c_old, A_b_old, b_b_old = pd_old[bits]
    A_c_new, b_c_new, per_class = pd_new[bits]
    A_k_c, b_k_c = per_class[c]
    assert torch.allclose(A_c_old, A_c_new),       f"bits={bits}: A_correct mismatch"
    assert torch.allclose(A_b_old, A_k_c),         f"bits={bits}: P3(c) vs old A_both mismatch"
print("\n→ Assertions passed: A_correct and P3(c) match old API exactly.")

New API — build_all_polytopes_per_class:
  A_base : (1929, 784)

  bits=4
    A_correct : (3849, 784)
    n_classes : 10
    P3(0): (3858, 784)  x∈P3(0)=False
    P3(1): (3858, 784)  x∈P3(1)=False
    P3(2): (3858, 784)  x∈P3(2)=True ← k* (q-pred)
    P3(3): (3858, 784)  x∈P3(3)=False
    P3(4): (3858, 784)  x∈P3(4)=False
    P3(5): (3858, 784)  x∈P3(5)=False
    P3(6): (3858, 784)  x∈P3(6)=False ← c (true class)
    P3(7): (3858, 784)  x∈P3(7)=False
    P3(8): (3858, 784)  x∈P3(8)=False
    P3(9): (3858, 784)  x∈P3(9)=False

  bits=8
    A_correct : (3849, 784)
    n_classes : 10
    P3(0): (3858, 784)  x∈P3(0)=False
    P3(1): (3858, 784)  x∈P3(1)=False
    P3(2): (3858, 784)  x∈P3(2)=False
    P3(3): (3858, 784)  x∈P3(3)=False
    P3(4): (3858, 784)  x∈P3(4)=False
    P3(5): (3858, 784)  x∈P3(5)=False
    P3(6): (3858, 784)  x∈P3(6)=True ← c (true class)
    P3(7): (3858, 784)  x∈P3(7)=False
    P3(8): (3858, 784)  x∈P3(8)=False
    P3(9): (3858, 784)  x∈P3(9)=False

→ Assertions pa

## 3 — run_volumes_v3k with N=3 directions

In [15]:
%%time
import sys
sys.path.insert(0, ROOT)

from scripts.run_volumes_v3k import run_volumes_v3k

N_CLASSES = 10
bounds    = [(-1., 1.)] * 784
n_workers = 4

result = run_volumes_v3k(
    A_base, b_base, pd_new, bounds,
    n_directions=N_DIR,
    n_workers=n_workers,
    n_classes=N_CLASSES,
)

print("\nRaw result:")
print(f"  n_directions_used : {result['n_directions_used']}")
print(f"  width_base (V1)   : {result['width_base']:.4f}")
for bits in BITS_TEST:
    wc = result['widths_correct'][bits]
    wb = result['widths_both'][bits]   # list of 10
    print(f"  bits={bits:2d}  V2={wc:.4f}  V3(k)={[f'{v:.4f}' for v in wb]}")

Directions: 100%|██████████| 2/2 [01:37<00:00, 48.99s/it]
Elapsed: 100.4s  |  Failed directions: 0/2
  width_base=39.2759
  bits= 4  width_correct=36.8209  widths_both=['0.0000', '0.0000', '36.8209', '0.0000', '0.0000', '0.0000', '36.7418', '0.0000', '0.0000', '0.0000']
  bits= 8  width_correct=39.1136  widths_both=['0.0000', '0.0000', '38.4399', '0.0000', '0.0000', '0.0000', '39.1136', '0.0000', '0.0000', '0.0000']



Raw result:
  n_directions_used : 2
  width_base (V1)   : 39.2759
  bits= 4  V2=36.8209  V3(k)=['0.0000', '0.0000', '36.8209', '0.0000', '0.0000', '0.0000', '36.7418', '0.0000', '0.0000', '0.0000']
  bits= 8  V2=39.1136  V3(k)=['0.0000', '0.0000', '38.4399', '0.0000', '0.0000', '0.0000', '39.1136', '0.0000', '0.0000', '0.0000']
CPU times: user 19.1 ms, sys: 55.7 ms, total: 74.8 ms
Wall time: 1min 40s


## 4 — Non-additivity check and GACC comparison

In [16]:
print(f"True class c = {c}")
print()

for bits in BITS_TEST:
    V2      = result['widths_correct'][bits]
    V3      = result['widths_both'][bits]          # list of 10
    V3_c    = V3[c]
    sum_V3  = sum(V3)
    kstar   = pred_qmodel[bits]

    gacc_old = V3_c / V2 if V2 > 0 else float('nan')
    gacc_new = V3_c / sum_V3 if sum_V3 > 0 else float('nan')

    print(f"bits={bits}  q-correct={q_correct[bits]}  q-pred={kstar}")
    print(f"  V1 (width_base)     = {result['width_base']:.4f}")
    print(f"  V2 (width_correct)  = {V2:.4f}")
    print(f"  V3(c={c})           = {V3_c:.4f}")
    if not q_correct[bits]:
        print(f"  V3(k*={kstar})         = {V3[kstar]:.4f}")
    print(f"  Σ_k V3(k)           = {sum_V3:.4f}   (expected ≈ {'1' if q_correct[bits] else '2'} × V2 = {V2 * (1 if q_correct[bits] else 2):.4f})")
    print(f"  Old GACC = V3(c)/V2 = {gacc_old:.4f}")
    print(f"  New GACC = V3(c)/Σ  = {gacc_new:.4f}   (expected ≈ {'1.0' if q_correct[bits] else '0.5'} for {'q-correct' if q_correct[bits] else 'q-incorrect'})")
    print()

True class c = 6

bits=4  q-correct=False  q-pred=2
  V1 (width_base)     = 39.2759
  V2 (width_correct)  = 36.8209
  V3(c=6)           = 36.7418
  V3(k*=2)         = 36.8209
  Σ_k V3(k)           = 73.5626   (expected ≈ 2 × V2 = 73.6417)
  Old GACC = V3(c)/V2 = 0.9979
  New GACC = V3(c)/Σ  = 0.4995   (expected ≈ 0.5 for q-incorrect)

bits=8  q-correct=True  q-pred=6
  V1 (width_base)     = 39.2759
  V2 (width_correct)  = 39.1136
  V3(c=6)           = 39.1136
  Σ_k V3(k)           = 77.5535   (expected ≈ 1 × V2 = 39.1136)
  Old GACC = V3(c)/V2 = 1.0000
  New GACC = V3(c)/Σ  = 0.5043   (expected ≈ 1.0 for q-correct)

